In [5]:
import geopandas as gpd
from tobler.area_weighted import area_interpolate

In [6]:
# NOCS/NAICS data
nn = gpd.read_file("../../data/census/ada-wide/toronto-tract-noc-naics.geojson")

# ADA wide data
ada = gpd.read_file("../../data/census/ada-wide/toronto-ada-wide.geojson")

In [7]:
# Areal interpolation

# Fields to interpolate (percentages -> intensive variables)
pct_fields = [
    "labour_creatives_pct",
    "labour_cultural_workers_pct",
    "labour_cultural_industries_pct",
    "labour_independent_artists_pct",
    "labour_arts_major_pct",
]

# Reproject to UTM 17N
nn = nn.to_crs(epsg=32617)
ada = ada.to_crs(epsg=32617)

# Fix invalid geometries
nn["geometry"] = nn.geometry.make_valid()
ada["geometry"] = ada.geometry.make_valid()

# Drop nn NA
nn_valid = nn.dropna(subset=pct_fields, how="all").copy()

# Interpolate (percentages are intensive -- averaged, weighted by area overlap)
result = area_interpolate(
    source_df=nn_valid,
    target_df=ada,
    intensive_variables=pct_fields,
)

# Attach to ada
for field in pct_fields:
    ada[field] = result[field].values.round(1)

# Project to WGS84 (EPSG:4326)
ada = ada.to_crs(epsg=4326)

In [8]:
# Write files
ada.to_file(
    "../../data/census/ada-wide/ada-wide-noc-naics.geojson", 
    driver="GeoJSON"
)

ada.to_file(
    "../../data/census/ada-wide/ada-wide-noc-naics.gpkg",
    layer="ada",
    driver="GPKG"
)

In [9]:
from IPython.core.display import display, HTML
display(HTML("<style>.output_scroll { height: auto !important; }</style>"))

/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_18548/1923396772.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [11]:
from IPython.core.display import display, HTML
display(HTML("<style>.output_scroll { height: auto !important; }</style>"))

# Print quantile breaks for tacLayerConfig.js
for field in pct_fields:
    quantiles = ada[field].quantile([0.2, 0.4, 0.6, 0.8])
    print(f"--- {field} ---")
    print(quantiles)

/var/folders/69/g590bg750s935v88zgfrdm9w0000gn/T/ipykernel_18548/1418930137.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


--- labour_creatives_pct ---
0.2    0.60
0.4    0.90
0.6    1.58
0.8    3.00
Name: labour_creatives_pct, dtype: float64
--- labour_cultural_workers_pct ---
0.2    1.0
0.4    1.7
0.6    2.5
0.8    5.3
Name: labour_cultural_workers_pct, dtype: float64
--- labour_cultural_industries_pct ---
0.2    0.90
0.4    1.30
0.6    1.98
0.8    4.20
Name: labour_cultural_industries_pct, dtype: float64
--- labour_independent_artists_pct ---
0.2    0.0
0.4    0.3
0.6    0.6
0.8    1.3
Name: labour_independent_artists_pct, dtype: float64
--- labour_arts_major_pct ---
0.2    2.30
0.4    3.52
0.6    5.10
0.8    8.24
Name: labour_arts_major_pct, dtype: float64


In [12]:
print(ada["labour_arts_major_pct"].quantile([0.2, 0.4, 0.6, 0.8]))

0.2    2.30
0.4    3.52
0.6    5.10
0.8    8.24
Name: labour_arts_major_pct, dtype: float64
